In [1]:
!git clone https://github.com/taoyds/test-suite-sql-eval

Cloning into 'test-suite-sql-eval'...


remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (14/14), done.


remote: Total 61 (delta 20), reused 16 (delta 16), pack-reused 31 (from 2)


Receiving objects: 100% (61/61), 619.62 KiB | 6.13 MiB/s, done.
Resolving deltas: 100% (24/24), done.


In [2]:
!pip install smolagents
!pip install sqlparse
!pip install sqlvalidator
!pip install json-repair

In [3]:
import shutil
try:
    shutil.rmtree("/kaggle/working/CISC839_A2")
except:
    pass
!git clone https://github.com/dcbisson23/CISC839_A2

Cloning into 'CISC839_A2'...


remote: Enumerating objects: 457, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (82/82), done.


remote: Total 457 (delta 55), reused 88 (delta 31), pack-reused 344 (from 1)
Receiving objects: 100% (457/457), 372.01 KiB | 4.18 MiB/s, done.


Resolving deltas: 100% (223/223), done.


In [4]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from transformers import set_seed
set_seed(20230334)

In [5]:
# This is mostly pulled from A1's py file, for ease of use.

# Kaggle
TABLES_PATH = "/kaggle/input/datasets/dcbisson23/spider-1-0/spider/tables.json"
QUESTIONS_PATH = "/kaggle/input/datasets/dcbisson23/spider-1-0/spider/dev.json"
GOLD_PATH = "/kaggle/input/datasets/dcbisson23/spider-1-0/spider/dev_gold.sql"
SPIDER_DB_PATH = "/kaggle/input/datasets/dcbisson23/spider-1-0/spider/database"
EXAMPLES_PATH = "/kaggle/input/datasets/dcbisson23/spider-1-0/spider/train_spider.json"
DATABASE_PATH = "/kaggle/input/datasets/dcbisson23/test-suite-sql-eval-database/database"
DATA_ROOT = "/kaggle/working/CISC839_A2/data"

# Local
# TABLES_PATH = "spider/tables.json"
# QUESTIONS_PATH = "spider/dev.json"
# GOLD_PATH = "spider/dev_gold.sql"
# SPIDER_DB_PATH = "spider/database"
# EXAMPLES_PATH = "spider/train_spider.json"
# DATABASE_PATH = "test-suite-sql-eval/database"
# DATA_ROOT = "data"

GEN_CONFIG_FILENAME = "generation_config.json"
PROMPTS_FILENAME = "formatted_prompts.jsonl"
RESPONSES_FILENAME = "responses.jsonl"
PREDICTIONS_FILENAME = "predictions.txt"
GOLD_FILENAME = "gold.txt"
AGENT_RESPONSES_FILENAME = "agent_responses.jsonl"
AGENT_VAR1_RESPONSES_FILENAME = "agent_var1_responses.jsonl"
AGENT_VAR2_RESPONSES_FILENAME = "agent_var2_responses.jsonl"

SAMPLE_RATE = 0.125
# Offset for the sampling of dev.json. Set to 0 for the set used in A1.
OFFSET = 0
# OFFSET = 1

# This is the system context used for prompting the non-agent LLM.
# NOTE: Modified from A1, removed clause restricting use of aliasing in query (found many gold examples used aliasing where A1 models did not).
SYS_CONTEXT = ("""You are an expert SQLite developer, tasked with writing a valid SQL query to answer a user question using a given SQL schema.
The user will provide the following information:
    1. The database schema as a standard SQLite DDL script, and
    3. The question to be answered by querying this database.
Your response must adhere to the following guidelines:
    1. (CRITICAL) You must provide ONLY a valid SQLite query to answer the provided question, and nothing else.
    2. Use ONLY the tables and columns provided in the schema.
    3. Do not use aliasing (x AS y) unless strictly necessary.
""")

# This is the model being used for A2.
MODEL = {
    "model": "Qwen/Qwen2.5-Coder-3B-Instruct"
         }

# Tools are called by providing a JSON blob in the following format:
# Action:
# {
#     "name": <tool name>,
#     "arguments": {
#         <arg_name>: <arg_value>,
#     }
# }
# Tools with multiple arguments should provide an <arg_name>: <arg_value> entry for each. If there are no arguments, leave the "arguments" field empty.

AGENT_SYS_PROMPT = """You are an expert SQLite developer.
Your task is to translate a natural-language question into a valid SQL query that will answer the question.

You will be provided the following information:
    1. The natural-language question to answer, and
    2. A database ID, which acts as a reference to the database.

You have access to the following tools:
{%- for tool in tools.values() %}
- {{ tool.to_tool_calling_prompt() }}
{%- endfor %}

The tool call you write is an action: after the tool is executed, you will get the result of the tool call as an "observation".
This Action/Observation loop can repeat N times. You will be provided all previous Actions and Observations at each step.
Use tool calls to collect information necessary to complete the task.

Your response must adhere to the following guidelines:
    1. (CRITICAL) Your response must consist of *exactly one* tool call, in the format specified above. If you do not provide a tool call or provide multiple tool calls, you will fail.
    2. Always use the right arguments for the tools. Never use variable names as the action arguments, use the value instead.
    3. (CRITICAL) To provide the final answer to the task, you MUST use the "final_answer" tool. It is the only way to complete the task, else you will be stuck on a loop.

Some best practices:
    1. Get the database schema before running any queries.
    2. Try and use all tools provided to you.
    3. Avoid making the same tool call (with the same arguments) more than once.

When calling the "final_answer" tool, the "answer" field MUST be a valid SQL query. Do NOT return the query's output. 
Only an answer in SQL query format will be accepted.
"""
AGENT_TASK = "Translate the natural-language question into an SQL query on the given database."

SENTENCE_TRANSFORMER = "sentence-transformers/all-MiniLM-L6-v2"
MAX_STEPS = 10

In [6]:
import os
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(f"{DATA_ROOT}/{MODEL['model']}", exist_ok=True)

In [7]:
# from CISC839_A2.scripts.utils import load_tokenizer, unload_model
# from CISC839_A2.scripts.generate_prompts import SpiderPromptBuilder

# builder = SpiderPromptBuilder(TABLES_PATH, QUESTIONS_PATH, sys_context=SYS_CONTEXT)

# tokenizer = load_tokenizer(MODEL)

# prompts_path = f"{DATA_ROOT}/{MODEL['model']}/{PROMPTS_FILENAME}"

# builder.generate_prompts(prompts_path, tokenizer, sample_rate=SAMPLE_RATE, offset=OFFSET)

# unload_model(tokenizer)

In [8]:
# # from CISC839_A2.scripts.utils import load_model, unload_model
# # from CISC839_A2.scripts.get_predictions import LLMPrompter

# # print("Loading model "+MODEL['model'])
# # model = load_model(MODEL)
# # tokenizer = load_tokenizer(MODEL)

# # gen_path = f"{DATA_ROOT}/{MODEL['model']}/{GEN_CONFIG_FILENAME}"
# # prompts_path = f"{DATA_ROOT}/{MODEL['model']}/{PROMPTS_FILENAME}"
# # responses_path = f"{DATA_ROOT}/{MODEL['model']}/{RESPONSES_FILENAME}"

# # prompter = LLMPrompter(model, tokenizer)
# # prompter.load_gen_config(gen_path)
# # prompter.run_prompt_pipeline(prompts_path, responses_path)

# # unload_model(model)
# # unload_model(tokenizer)

# import json
# from CISC839_A2.scripts.utils import load_tokenizer, get_data_from_response
# responses_path = f"{DATA_ROOT}/{MODEL['model']}/{RESPONSES_FILENAME}"
# predictions_path = f"{DATA_ROOT}/{MODEL['model']}/{PREDICTIONS_FILENAME}"
# gold_path = f"{DATA_ROOT}/{MODEL['model']}/{GOLD_FILENAME}"

# with open(responses_path) as f:
#     responses = f.readlines()
# with open(GOLD_PATH) as f:
#     gold_sql = f.readlines()


# predictions = []
# golds = []
# reasoning_traces = []

# for response_json in responses:
#     response_dict = json.loads(response_json)
#     id = response_dict["id"]
#     db_id = response_dict["db_id"]
#     response = response_dict["response"]
#     response_data = get_data_from_response(response)
#     pred = response_data["prediction"]
#     gold = gold_sql[id]
#     predictions.append(pred)
#     # To match needed formatting for evaluation script.
#     golds.append(gold)

# with open(gold_path, "w", encoding="utf-8") as f_out:
#     f_out.write("".join(golds))
# with open(predictions_path, "w", encoding="utf-8") as f_out:
#     f_out.write("\n".join(predictions))

# import os
# import sys
# cwd = os.getcwd()
# sys.path.insert(0, f"{cwd}/test-suite-sql-eval")

# # Run this for a deeper score analysis on a question-by-question basis.
# from process_sql import Schema, get_schema, get_sql
# from evaluation import Evaluator
# from exec_eval import eval_exec_match
# import json
# import csv

# import nest_asyncio
# nest_asyncio.apply()

# predictions_path = f"{DATA_ROOT}/{MODEL['model']}/{PREDICTIONS_FILENAME}"
# gold_path = f"{DATA_ROOT}/{MODEL['model']}/{GOLD_FILENAME}"
# database_path = f"/kaggle/input/datasets/dcbisson23/test-suite-sql-eval-database/database"

# with open(gold_path, "r") as f:
#     # Took this code from evaluation.py.evaluate.
#     glist = []
#     gseq_one = []
#     for l in f.readlines():
#         if len(l.strip()) == 0:
#             glist.append(gseq_one)
#             gseq_one = []
#         else:
#             lstrip = l.strip().split('\t')
#             gseq_one.append(lstrip)
#     if len(gseq_one) != 0:
#         glist.append(gseq_one)
#     # End of block
# # Not sure why but they nest glist in another list so I'm changing that here.
# glist = glist[0]
# evaluator = Evaluator()
# results = []
# with open(predictions_path, "r") as f:
#     for idx, p_str in enumerate(f.readlines()):
#         # Can't use question id for this since using gold
#         g = glist[idx]
#         # More taken from eval script.
#         g_str, db_id = g
#         db = os.path.join(database_path, db_id, db_id + ".sqlite")
#         schema = Schema(get_schema(db))
#         g_sql = get_sql(schema, g_str)
#         hardness = evaluator.eval_hardness(g_sql)
#         score = eval_exec_match(
#             db=db,
#             p_str=p_str,
#             g_str=g_str,
#             plug_value=False,
#             keep_distinct=False,
#             progress_bar_for_each_datapoint=False
#         )
#         result = {
#             "id": idx,
#             "db_id": db_id,
#             "hardness": hardness,
#             "score": score,
#         }
#         results.append(result)

# results_path = f"{DATA_ROOT}/{MODEL['model']}/full_results.csv"
# with open(results_path, "w", encoding="utf-8", newline="") as f_out:
#     fieldnames = ["id", "db_id", "hardness", "score"]
#     writer = csv.DictWriter(f_out, fieldnames=fieldnames)
#     writer.writeheader()
#     writer.writerows(results)


# from scripts.utils import get_data_from_response
# import json
# import csv

# responses_path = f"{DATA_ROOT}/{MODEL['model']}/{RESPONSES_FILENAME}"
# gold_path = f"{DATA_ROOT}/{MODEL['model']}/{GOLD_FILENAME}"
# results_path = f"{DATA_ROOT}/{MODEL['model']}/full_results.csv"

# with open(QUESTIONS_PATH, "r") as f:
#     questions = json.load(f)
# with open(responses_path, "r") as f:
#     responses = f.readlines()
# with open(gold_path, "r") as f:
#     golds = f.readlines()
# with open(results_path, "r") as f:
#     reader = csv.DictReader(f)
#     results = [row for row in reader]


# bad_items = []
# for result in results:
#     if int(result["score"]) == 0:
#         response_json = responses[int(result["id"])]
#         response_dict = json.loads(response_json)
#         qid = response_dict["id"]
#         question = questions[qid]["question"]
#         response = get_data_from_response(response_dict["response"])["prediction"]
#         bad_item = {
#             "id": qid,
#             "db_id": result["db_id"],
#             "hardness": result["hardness"],
#             "question": question,
#             "prediction": predictions[int(result["id"])],
#             "gold": golds[int(result["id"])]
#         }
#         bad_items.append(bad_item)
# bad_results_path = f"{DATA_ROOT}/{MODEL['model']}/bad_predictions.csv"
# with open(bad_results_path, "w", encoding="utf-8", newline="") as f_out:
#     fieldnames = ["id", "db_id", "hardness", "question", "prediction", "gold"]
#     writer = csv.DictWriter(f_out, fieldnames=fieldnames)
#     writer.writeheader()
#     writer.writerows(bad_items)


In [9]:
from sentence_transformers import SentenceTransformer
from CISC839_A2.scripts.agents import AgentHarness, QueryTool, GetSchemaTool, TableOrdererTool, ExampleRetrieverTool
from smolagents import FinalAnswerTool

sentence_transformer = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
query_tool = QueryTool(db_root=SPIDER_DB_PATH)
schema_tool = GetSchemaTool(db_root=SPIDER_DB_PATH)
table_tool = TableOrdererTool(db_root=SPIDER_DB_PATH, encoder=sentence_transformer)
example_tool = ExampleRetrieverTool(encoder=sentence_transformer, examples_path=EXAMPLES_PATH)
final_answer_tool = FinalAnswerTool()

In [11]:
from CISC839_A2.scripts.utils import load_gen_config, get_questions
questions = get_questions(questions_path=QUESTIONS_PATH, sample_rate=SAMPLE_RATE, offset=OFFSET)
gen_config = load_gen_config(gen_config_path=f"{DATA_ROOT}/{MODEL['model']}/{GEN_CONFIG_FILENAME}")

Generating prompts: 0it [00:00, ?it/s]

Generating prompts: 1034it [00:00, 1342281.13it/s]

In [12]:
def get_response_data(agent, f_out):
    agent.agent.memory.reset()
    result = agent.run(question=question_dict["question"], 
                       db_id=question_dict["db_id"], 
                       task=AGENT_TASK,
                       max_steps=MAX_STEPS,
                       return_full_result=True)
    steps = []
    for step in agent.agent.memory.steps:
        step_dict = {}
        if isinstance(step, TaskStep):
            step_dict['task'] = step.task
        elif isinstance(step, ActionStep):
            step_dict['step'] = step.step_number
            step_dict['action'] = step.action_output
            step_dict['observations'] = step.observations
            step_dict['error'] = str(step.error)
        steps.append(step_dict)
    record = {
        "output": result.output,
        "state": result.state,
        "steps": steps,
        "token_usage": result.token_usage.dict(),
        "timing": result.timing.dict()
    }
    f_out.write(json.dumps(record) + "\n")

In [13]:
from smolagents import TaskStep, ActionStep
import json
from tqdm import tqdm
import gc
import torch

agent = AgentHarness(
    model_id=MODEL["model"],
    gen_config=gen_config,
    sys_prompt=AGENT_SYS_PROMPT,
    tools=[schema_tool, 
           query_tool, 
           table_tool, 
           example_tool,
           final_answer_tool],
    verbosity=-1
)
torch.cuda.memory._record_memory_history(max_entries=100000)
with open(f"{DATA_ROOT}/{MODEL['model']}/{AGENT_RESPONSES_FILENAME}", "w", encoding="utf8") as f_out:
    for idx, question_dict in tqdm(enumerate(questions), desc="Evaluating questions with AgentHarness"):
        try:
            data = get_response_data(agent, f_out)
            del data
        except:
            pass
        gc.collect()
        torch.cuda.empty_cache()
torch.cuda.memory._dump_snapshot("/kaggle/working/memory_snapshot.pickle")
torch.cuda.memory._record_memory_history(enabled=None)
del agent
gc.collect()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

... (139 outputs left out)

Evaluating questions with AgentHarness: 134it [3:29:21, 40.08s/it]

Evaluating questions with AgentHarness: 135it [3:30:53, 55.87s/it]

Evaluating questions with AgentHarness: 135it [3:30:53, 93.73s/it]

10658

In [14]:


agent_var1 = AgentHarness(
    model_id=MODEL["model"],
    gen_config=gen_config,
    sys_prompt=AGENT_SYS_PROMPT,
    tools=[query_tool, 
           schema_tool, 
           example_tool,
           final_answer_tool],
    verbosity=-1
)
with open(f"{DATA_ROOT}/{MODEL['model']}/{AGENT_VAR1_RESPONSES_FILENAME}", "w", encoding="utf8") as f_out:
    for idx, question_dict in tqdm(enumerate(questions), desc="Evaluating questions with AgentHarness"):
        try:
            data = get_response_data(agent_var1, f_out)
            del data
        except:
            pass
        gc.collect()
        torch.cuda.empty_cache()
del agent_var1
gc.collect()

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Evaluating questions with AgentHarness: 0it [00:00, ?it/s]

Evaluating questions with AgentHarness: 1it [03:17, 197.14s/it]

Evaluating questions with AgentHarness: 2it [12:08, 393.75s/it]

Evaluating questions with AgentHarness: 3it [15:38, 309.81s/it]

Evaluating questions with AgentHarness: 4it [23:08, 365.25s/it]

... (79 outputs left out)

Evaluating questions with AgentHarness: 82it [5:52:19, 151.25s/it]

Evaluating questions with AgentHarness: 83it [5:56:34, 182.41s/it]

Evaluating questions with AgentHarness: 84it [6:02:08, 227.73s/it]

Evaluating questions with AgentHarness: 85it [8:25:39, 2742.96s/it]

In [ ]:
agent_var2 = AgentHarness(
    model_id=MODEL["model"],
    gen_config=gen_config,
    sys_prompt=AGENT_SYS_PROMPT,
    tools=[query_tool, 
           schema_tool, 
           table_tool,
           final_answer_tool],
    verbosity=-1
)
with open(f"{DATA_ROOT}/{MODEL['model']}/{AGENT_VAR2_RESPONSES_FILENAME}", "w", encoding="utf8") as f_out:
    for idx, question_dict in tqdm(enumerate(questions), desc="Evaluating questions with AgentHarness"):
        try:
            data = get_response_data(agent_var2, f_out)
            del data
        except:
            pass
        gc.collect()
        torch.cuda.empty_cache()
del agent_var2
gc.collect()